# 🧪 Laboratório Prático: Alucinação como Consequência Estrutural e Mecanismos de Mitigação
**Disciplina:** COM170 — Inteligência Artificial na Prática Acadêmica e Profissional  
**Quinzena 02:** Prompts, Atenção, Alucinação e Inferência  
**Perfil:** Estudo Prático Guiado (Passo a Passo)

---

## 🎯 Objetivos de Aprendizagem
Neste laboratório, você investigará a raiz probabilística da alucinação e estratégias de contenção:

1. **Causa Raiz Probabilística da Alucinação:** Por que modelos de linguagem "inventam" fatos com alta confiança aparente.
2. **Alucinação Intrínseca vs. Extrínseca:** Compreensão prática dos dois modos fundamentais de erro factual com estudos de caso reais (*Mata v. Avianca*).
3. **A Taxonomia dos 5 Tipos de Erro:** Reproduzir e diagnosticar na prática: *Alucinação Factual*, *Viés de Corpus*, *Vagueza Confortável*, *Coerência sem Verdade (Sofisma)* e *Ausência de Contexto*.
4. **Estratégias de Mitigação e Autoria Responsável:** Testar a técnica de *Context Grounding* (ancoragem em texto) para conter alucinações.

---

## ⚙️ Pré-requisitos: Isolamento com Ambiente Virtual via `uv`
Para manter seu Python global limpo, o projeto utiliza o `uv` na raiz do repositório para gerenciar o ambiente virtual e as dependências:

### 1. Instalar as Dependências com `uv`
Abra o terminal na raiz do repositório (`EngComp-UNIVESP`) e execute:
```bash
uv add torch transformers ipykernel
```

### 2. Selecionar o Kernel no VS Code / Jupyter
> 💡 **Dica de execução:** No canto superior direito deste notebook no VS Code, clique em **Select Kernel** (ou *Selecionar Kernel*) $\rightarrow$ **Python Environments...** $\rightarrow$ selecione o interpretador `.venv` criado pelo `uv` na raiz.


## 📦 Etapa 1: Importação das Bibliotecas e Configuração do Ambiente

Importamos o PyTorch (`torch`), funções matemáticas (`torch.nn.functional`) e as classes do Hugging Face `transformers`.

In [ ]:
import time
import math
import torch
import torch.nn.functional as F
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    GPT2Tokenizer,
    GPT2Model,
    GPT2LMHeadModel
)

print(f"Versão do PyTorch: {torch.__version__}")
print(f"Dispositivo disponível para execução: {'CUDA (GPU)' if torch.cuda.is_available() else 'CPU'}")


---
## 🧠 Etapa 2: A Causa Raiz da Alucinação (Por que o modelo não verifica a verdade?)

Para entender **por que** a alucinação ocorre, precisamos olhar a função objetivo matemática que treinou o modelo:

$$\mathcal{L}_{\text{treino}} = -\sum_{t=1}^{N} \log P(w_t \mid w_1, w_2, \dots, w_{t-1})$$

O modelo é otimizado exclusivamente para **prever o próximo token mais plausível** com base no padrão estatístico dos textos do corpus de treino.

### ⚠️ Implicações Arquiteturais Fundamentais:
1. **Não há conexão com bancos de dados relacionais:** O modelo não executa queries SQL, nem consulta a internet durante a inferência local.
2. **Não existe função de checagem de veracidade:** O algoritmo avalia se uma frase "soa fluente e provável", e não se ela corresponde a um fato do mundo real.
3. **Alucinação é estrutural, não um bug passageiro:** Enquanto a função de perda for probabilística pura, sequências que aparentam verdade sempre serão geradas.

Vamos calcular as probabilidades atribuídas a frases factualmente falsas mas com sintaxe impecável para observar esse fenômeno numericamente.

In [ ]:
tokenizador_gpt2 = GPT2Tokenizer.from_pretrained("gpt2")
modelo_gpt2 = GPT2LMHeadModel.from_pretrained("gpt2")
# Vamos testar uma frase com sintaxe perfeita, mas com fato inventado
prompt_teste = "The famous 19th-century treaty was signed in Paris on"
tokens_prompt = tokenizador_gpt2(prompt_teste, return_tensors="pt")
with torch.no_grad():
    saida = modelo_gpt2(**tokens_prompt)
# Obter logits e probabilidades do próximo token
logits = saida.logits[0, -1, :]
probabilidades = F.softmax(logits, dim=-1)
top_probabilidades, top_indices = torch.topk(probabilidades, 5)
print(f"Prompt: '{prompt_teste}'")
print("\nTop-5 continuações estatisticamente mais prováveis para o modelo:")
for posicao, (probabilidade, indice_token) in enumerate(zip(top_probabilidades, top_indices), start=1):
    token_texto = tokenizador_gpt2.decode([indice_token.item()])
    print(f"  #{posicao}: Token '{token_texto:<10}' | Probabilidade calculada: {probabilidade.item() * 100:.2f}%")
print("\n🔍 OBSERVAÇÃO DIDÁTICA:")
print("O modelo prioriza meses e datas genéricas por associação estatística de tratados históricos,")
print("sem saber se o 'tratado do século XIX' a que você se refere de fato existiu naquele dia.")


---
## ⚖️ Etapa 3: Alucinação Extrínseca e o Caso *Mata v. Avianca (2023)*

A pesquisa de **Ji et al. (2023)** e **Bang et al. (2023)** classifica as alucinações em duas categorias:

1. **Alucinação Intrínseca (de dentro):** O modelo contradiz uma informação explicitamente fornecida no prompt (ex.: recebe um texto afirmando que o lucro aumentou, mas resume dizendo que caiu).
2. **Alucinação Extrínseca (de fora):** O modelo inventa dados, citações e fatos que não existem em lugar nenhum, apoiando-se apenas na sua **memória paramétrica** (pesos congelados).

> 🚨 **Caso Real (Mata v. Avianca):** Advogados em Nova York usaram o ChatGPT para redigir uma petição judicial. O modelo gerou 6 decisões judiciais completas com nomes, números de processo e citações impecáveis — **nenhuma delas existia no mundo real**. Os advogados foram multados em US$ 5.000 pelo juiz federal Kevin Castel.

Vamos testar a geração livre da memória paramétrica pedindo referências acadêmicas sobre um tema inexistente:

In [ ]:
nome_modelo_qwen = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizador_qwen = AutoTokenizer.from_pretrained(nome_modelo_qwen)
modelo_qwen = AutoModelForCausalLM.from_pretrained(nome_modelo_qwen)
# Teste de Alucinação Extrínseca (Memória Paramétrica pura)
prompt_citacao_ficticia = "Cite 2 artigos científicos publicados em 2024 sobre o algoritmo quântico de Silva-Ferreira para redes neurais:"
entradas_tokenizadas_citacao = tokenizador_qwen(prompt_citacao_ficticia, return_tensors="pt")
torch.manual_seed(42)
with torch.no_grad():
    saida_citacao = modelo_qwen.generate(
        **entradas_tokenizadas_citacao,
        max_new_tokens=60,
        do_sample=True,
        temperature=0.7,
        pad_token_id=tokenizador_qwen.eos_token_id
    )
resposta_citacao = tokenizador_qwen.decode(saida_citacao[0], skip_special_tokens=True)
print("==================================================")
print("📌 TESTE DE MEMÓRIA PARAMÉTRICA (Alucinação Extrínseca)")
print("==================================================")
print(resposta_citacao)
print("==================================================")
print("🔍 Diagnóstico: O algoritmo 'Silva-Ferreira' não existe, mas o modelo gera títulos plausíveis,")
print("revistas fictícias e formatação acadêmica padrão para satisfazer a continuidade do texto.")


---
## 🛡️ Etapa 4: Mitigação com Ancoragem de Contexto (*Context Grounding / RAG*)

Como vimos no Módulo 3 da apostila, a forma mais eficaz de proteger um LLM contra alucinações extrínsecas é **fornecer o texto de referência diretamente no prompt** (ancoragem de contexto / *grounding*).

Vamos comparar a resposta do modelo quando ele é forçado a consultar apenas o texto fornecido:

In [ ]:
contexto_documento = """DOCUMENTO DE REFERÊNCIA:
O projeto Helios-2026 foi desenvolvido pelo Laboratório de Computação da UNIVESP.
O orçamento total aprovado foi de R$ 45.000,00 e o prazo de entrega final é 15 de dezembro de 2026.
Qualquer informação não contida neste documento deve ser declarada como 'Não disponível no texto'."""
pergunta_ancorada = f"""{contexto_documento}
PERGUNTA: Qual é o orçamento do projeto Helios-2026 e quem é o diretor geral do projeto?
RESPONDA APENAS COM BASE NO DOCUMENTO:"""
entradas_tokenizadas_ancoradas = tokenizador_qwen(pergunta_ancorada, return_tensors="pt")
with torch.no_grad():
    saida_ancorada = modelo_qwen.generate(
        **entradas_tokenizadas_ancoradas,
        max_new_tokens=60,
        do_sample=False,  # Geração determinística (Greedy)
        pad_token_id=tokenizador_qwen.eos_token_id
    )
resposta_ancorada = tokenizador_qwen.decode(saida_ancorada[0], skip_special_tokens=True)
print("==================================================")
print("🛡️ RESPOSTA COM ANCORAGEM DE CONTEXTO (Grounding)")
print("==================================================")
print(resposta_ancorada[len(pergunta_ancorada):].strip())
print("==================================================")


---
## 🔬 Etapa 5: Investigação Prática da Taxonomia dos 5 Tipos de Erro

A apostila da Quinzena 02 estabelece uma taxonomia prática com cinco categorias de erro:

| Tipo de Erro | Análogo Humano | O que é | Exemplo Prático |
| :--- | :--- | :--- | :--- |
| **1. Alucinação Factual** | *O Blefe* | Afirmação falsa com convicção e dados fictícios. | Citar artigos ou leis inexistentes. |
| **2. Viés de Corpus** | *O Preconceito Automático* | Reprodução de correlações estatísticas históricas dos dados de treino. | Associar profissões a estereótipos demográficos. |
| **3. Vagueza Confortável** | *A Evasão* | Resposta prolixa e genérica que não se compromete com fatos concretos. | Listar prós e contras genéricos sem responder a pergunta. |
| **4. Coerência sem Verdade** | *O Sofisma* | Argumento formalmente elegante com premissa ou conclusão falsa. | Dedução lógica impecável baseada em conceitos errôneos. |
| **5. Ausência de Contexto** | *A Generalização* | Resposta genérica internacional quando a demanda era específica local. | Recomendar normas técnicas dos EUA para obra no Brasil. |

Vamos inspecionar um exemplo de **Coerência sem Verdade (Sofisma)** e de **Ausência de Contexto**:

In [ ]:
# Teste de Ausência de Contexto
prompt_norma = "Qual norma técnica regulamenta a instalação de disjuntores residenciais?"
entradas_tokenizadas_norma = tokenizador_qwen(prompt_norma, return_tensors="pt")
with torch.no_grad():
    saida_norma = modelo_qwen.generate(
        **entradas_tokenizadas_norma,
        max_new_tokens=50,
        temperature=0.3,
        do_sample=True,
        pad_token_id=tokenizador_qwen.eos_token_id
    )
resposta_norma = tokenizador_qwen.decode(saida_norma[0], skip_special_tokens=True)
print("==================================================")
print("🔍 DIAGNÓSTICO: Ausência de Contexto (A Generalização)")
print("==================================================")
print(resposta_norma)
print("--------------------------------------------------")
print("💡 Lição: Como o prompt não especificou 'no Brasil (normas ABNT/NBR 5410)', o modelo")
print("tende a responder com normas internacionais (IEC, NEC ou genéricas) devido à prevalência no corpus.")
print("==================================================")


---
## 📊 Etapa 6: Matriz de Risco e Prática de Autoria Acadêmica Responsável

Como aponta o estudo de **Bang et al. (2023)**, a taxa de acurácia média em tarefas de raciocínio factual gira em torno de **63,41%** (cerca de 1 erro a cada 3 respostas complexas).

### 📋 Matriz de Verificação para o Estudante de Engenharia da Computação:
| Nível de Risco | Cenário de Uso | Ação Exigida |
| :---: | :--- | :--- |
| **Baixo** | Brainstorming, ideação e estruturação inicial de código. | Uso livre das ideias geradas; não exige validação de fontes. |
| **Médio** | Revisão de conceitos de programação e sintaxe. | Validar termos técnicos na documentação oficial da linguagem. |
| **Alto** | Pesquisa acadêmica, citações de artigos e normas técnicas. | **Obrigatório:** Localizar o DOI, autor e texto original no Google Acadêmico/Scopus. |
| **Crítico / Máximo** | Cálculos de engenharia, segurança e decisões profissionais. | **Nunca usar o modelo como fonte primária.** Conferir em normas ABNT e fontes oficiais. |

---

## 📝 Roteiro de Fixação e Autoavaliação
1. **Por que o modelo não mente?** Qual é a diferença epistemológica entre um ser humano *mentir* e um LLM *alucinar*?
2. **Memória Paramétrica:** Por que modelos de linguagem cometem significativamente mais alucinações extrínsecas quando respondem sem contexto fornecido?
3. **Responsabilidade Autoral:** No caso *Mata v. Avianca*, por que a responsabilidade jurídica e ética recaiu integralmente sobre os advogados e não sobre a OpenAI/ChatGPT?

---

### 🏷️ Conexão com o Mundo Real: Marca d'Água Estatística, Regulação (AI Act) e Autoria
Recentemente, a Anthropic anunciou que todos os novos modelos do Claude virão com uma marca d’água invisível e legível por máquinas para cumprir o Artigo 50(2) do *AI Act* europeu (junto a OpenAI, Meta, Microsoft e Google).

* **Mecanismo Estatístico:** O modelo divide o vocabulário em tokens "verdes" e "vermelhos" com uma chave secreta e inclina sutilmente as probabilidades de escolha durante a geração. Em textos longos, um detector com a chave enxerga um padrão que não existe na natureza.
* **Impacto Crítico na Autoria Acadêmica:** Mesmo textos escritos originalmente por humanos que passarem pelo modelo para uma simples edição ou revisão podem voltar com a assinatura de "passou pelo modelo". Isso reforça a necessidade de responsabilidade autoral e rigor metodológico no uso de assistentes de IA na universidade.
